# Storage Drivers — 02: DuckDB Analytical OTF Queries

**Persona:** Data Analyst

**Goal:** Add `driver:duckdb` as a secondary READ/SEARCH driver backed by a GCS Parquet
file, configure routing so aggregation queries go through DuckDB while writes and
transactional reads stay on PostgreSQL, then run an aggregation.

---

## Prerequisites

- DynaStore running with the `driver:duckdb` extra installed:
  `pip install dynastore[duckdb]`
- A GCS Parquet file is accessible at `GCS_PARQUET_PATH` (e.g. `gs://my-bucket/data.parquet`)
- GCS credentials are available in the environment (`GOOGLE_APPLICATION_CREDENTIALS` or
  Workload Identity) — DuckDB will not silently retry without them
- `driver:postgresql` is already configured on the collection (run notebook 01 first)

## Environment variables

| Variable | Default | Description |
|---|---|---|
| `DYNASTORE_BASE_URL` | `http://localhost:8080` | API base URL |
| `DYNASTORE_ADMIN_TOKEN` | _(empty)_ | Bearer token for admin operations |
| `CATALOG_ID` | `demo-catalog` | Target catalog |
| `COLLECTION_ID` | `sentinel2-l2a` | Target collection |
| `DUCKDB_POOL_SIZE` | `3` | Max concurrent DuckDB connections |
| `GCS_PARQUET_PATH` | `gs://my-bucket/data.parquet` | GCS path to the Parquet dataset |

In [ ]:
import os
import json
import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL        = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
ADMIN_TOKEN     = os.environ.get("DYNASTORE_ADMIN_TOKEN", "")
CATALOG_ID      = os.environ.get("CATALOG_ID", "demo-catalog")
COLLECTION_ID   = os.environ.get("COLLECTION_ID", "sentinel2-l2a")
DUCKDB_POOL_SIZE = os.environ.get("DUCKDB_POOL_SIZE", "3")
GCS_PARQUET_PATH = os.environ.get("GCS_PARQUET_PATH", "gs://my-bucket/data.parquet")

headers = {"Authorization": f"Bearer {ADMIN_TOKEN}"} if ADMIN_TOKEN else {}
client  = httpx.Client(base_url=BASE_URL, headers=headers, timeout=60)

print(f"Base URL         : {BASE_URL}")
print(f"Catalog          : {CATALOG_ID}")
print(f"Collection       : {COLLECTION_ID}")
print(f"DuckDB pool size : {DUCKDB_POOL_SIZE}")
print(f"GCS Parquet path : {GCS_PARQUET_PATH}")
print(f"Auth header      : {'set' if ADMIN_TOKEN else 'not set'}")

In [ ]:
# Step 1 — Inspect the driver:duckdb JSON schema
resp = client.get("/configs/schemas/CollectionDuckdbDriverConfig")
assert resp.status_code == 200, (
    f"Expected 200, got {resp.status_code}: {resp.text[:400]}"
)

schema = resp.json()
properties = schema.get("properties", schema.get("schema", {}).get("properties", {}))

print("driver:duckdb schema (selected fields):")
for field in ("enabled", "parquet_path", "pool_size"):
    if field in properties:
        print(f"  {field}: {json.dumps(properties[field], indent=4)}")
    else:
        print(f"  {field}: (not found — check API version or driver extra)")

In [ ]:
# Step 2 — Configure driver:duckdb
#
# pool_size caps the number of concurrent DuckDB connections (bounded connection pool).
# DuckDB is async-safe via run_in_thread; each connection is held exclusively during
# a query and returned to the pool on completion. Set pool_size to match expected
# concurrent analytical query load (not overall request concurrency).
DUCKDB_CONFIG = {
    "enabled": True,
    "parquet_path": GCS_PARQUET_PATH,
    "pool_size": int(DUCKDB_POOL_SIZE),
}

resp = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/CollectionDuckdbDriverConfig",
    json=DUCKDB_CONFIG,
)
assert resp.status_code == 200, (
    f"Expected 200, got {resp.status_code}: {resp.text[:400]}"
)

saved = resp.json()
print("driver:duckdb config saved:")
print(json.dumps(saved, indent=2))

In [ ]:
# Step 3 — Set routing: SEARCH→duckdb, WRITE/READ→postgresql
#
# DuckDB is a read-only analytical engine and does NOT have the TRANSACTIONS
# capability. The platform will reject any routing config that places duckdb
# in a WRITE slot. Placing it in SEARCH only is the correct pattern.
ROUTING_CONFIG = {
    "enabled": True,
    "operations": {
        "WRITE":  [{"driver_id": "CollectionPostgresqlDriver", "on_failure": "fatal"}],
        "READ":   [{"driver_id": "CollectionPostgresqlDriver", "on_failure": "fatal"}],
        "SEARCH": [{"driver_id": "CollectionDuckdbDriver",     "on_failure": "fatal"}],
    },
}

resp = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/CollectionRoutingConfig",
    json=ROUTING_CONFIG,
)
# 200 on first configuration; 409 if identical config was already persisted
# (the routing `operations` field is Immutable, so any PUT that doesn't change
# the structural shape is idempotent from the user's perspective).
assert resp.status_code in (200, 409), (
    f"Expected 200 or 409, got {resp.status_code}: {resp.text[:400]}"
)

if resp.status_code == 200:
    print("Routing config saved:")
    print(json.dumps(resp.json(), indent=2))
else:
    print("Routing already matches (409 Immutable) — no-op.")

In [ ]:
# Step 4 — Run an aggregation through the DuckDB SEARCH driver
#
# POST /stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/aggregate
# uses the AggregationRequest body (OGC API - Features Part 3 / STAC Aggregation extension).
# This endpoint may return 501 Not Implemented on deployments where the aggregation
# extension is not enabled — that is handled gracefully below.
# AggregationRule schema (from dynastore.modules.stac.stac_config):
#   type       : one of term|stats|geohash|geotile|datetime|bbox|temporal_extent
#   property   : dotted path into the item (e.g. 'properties.eo:cloud_cover')
#   interval   : optional — for type=datetime (e.g. '1d', '1M', '1y')
aggregation_request = {
    "aggregations": [
        {"name": "datetime_frequency", "type": "datetime",
         "property": "properties.datetime", "interval": "1M"},
        {"name": "cloud_cover_stats",  "type": "stats",
         "property": "properties.eo:cloud_cover"},
    ],
}

resp = client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/aggregate",
    json=aggregation_request,
)

if resp.status_code == 200:
    agg_result = resp.json()
    print("Aggregation result:")
    print(json.dumps(agg_result, indent=2))
elif resp.status_code == 501:
    # Aggregation extension is optional. If not enabled this is expected.
    print("501 Not Implemented — aggregation extension not enabled on this deployment.")
    print("Enable it by installing dynastore[aggregation] and adding AggregationExtension")
    print("to the application's extension list.")
else:
    assert False, f"Unexpected status {resp.status_code}: {resp.text[:400]}"

## Edge cases

### Case A — DuckDB in the WRITE routing slot

`driver:duckdb` does not have the `TRANSACTIONS` capability. The platform's capability
gate should reject any routing config that places it in a WRITE slot.
The cell below shows what such a config looks like and the expected rejection.

In [ ]:
# DuckDB in WRITE slot — expected to be rejected by capability gate
invalid_routing = {
    "enabled": True,
    "operations": {
        # DuckDB is read-only; placing it in WRITE must be rejected
        "WRITE": [{"driver_id": "CollectionDuckdbDriver",     "on_failure": "fatal"}],
        "READ":  [{"driver_id": "CollectionPostgresqlDriver", "on_failure": "fatal"}],
    },
}

resp = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/CollectionRoutingConfig",
    json=invalid_routing,
)

print(f"Invalid routing (DuckDB in WRITE) → {resp.status_code}")
if resp.status_code in (400, 409, 422):
    detail = resp.json().get("detail", resp.text[:300])
    print(f"  Rejected as expected: {detail}")
elif resp.status_code == 200:
    # Some deployments validate capability only at write time, not config time.
    # Document which behavior you observe.
    print("  WARNING: config was accepted — capability gate may be enforced at write time.")
    print("  Attempting a write will fail with a TRANSACTIONS capability error.")
    # Restore the valid routing
    client.put(
        f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/CollectionRoutingConfig",
        json=ROUTING_CONFIG,
    )
else:
    print(f"  Unexpected status: {resp.text[:300]}")

In [ ]:
# GCS path without credentials — expected explicit error, not a silent hang
#
# If GOOGLE_APPLICATION_CREDENTIALS is not set and the DuckDB httpfs extension
# cannot resolve credentials from the environment (Workload Identity, etc.),
# DuckDB will raise an IOException when the Parquet scan is opened.
# DynaStore surfaces this as a 500 or 503 with an error message containing
# "GCS" or "credentials" — it will NOT hang waiting for a retry.
#
# To reproduce: temporarily unset GOOGLE_APPLICATION_CREDENTIALS and run a SEARCH.
# The error should appear within the request timeout (default 30 s).
print("GCS credential error behavior (documentation only — not executed):")
print()
print("  Symptom : POST /aggregate returns 500 within 30 s")
print("  Message : 'IOException: failed to open GCS file <path>: ...'")
print("  Fix     : Set GOOGLE_APPLICATION_CREDENTIALS or run on a GCP VM")
print("            with Workload Identity attached to the service account.")
print()
print("  A silent hang (no response for minutes) indicates the httpfs extension")
print("  is attempting a credential refresh loop — upgrade duckdb-httpfs.")

## Teardown

Remove the `driver:duckdb` config. Routing is left pointing to PostgreSQL for
READ/WRITE; the SEARCH slot will fall back to the platform default.

In [ ]:
resp = client.delete(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/CollectionDuckdbDriverConfig"
)
assert resp.status_code == 204, (
    f"Expected 204 No Content, got {resp.status_code}: {resp.text[:400]}"
)
print(f"Deleted driver:duckdb config — status {resp.status_code}")

client.close()